In [6]:
from umap import UMAP
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, RobustScaler
from sklearn.utils import resample
from sklearn.metrics import pairwise_distances
from scipy.spatial.transform import Rotation as R
from pyDRMetrics.pyDRMetrics import *
from itertools import product
from joblib import Parallel, delayed
from pathlib import Path
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.io as pio
pio.renderers.default = "browser"


# Change working directory
import os
os.chdir(os.getcwd() + "/../")

import config
import hyp_config
from visualisation.plotting import plot_embedding, plot_interactive_embedding
# from utils.analysis import rotate_umap, get_user_input

# UMAP hyperparameter investigation
The parameters were selected such that the dimensionality-reduced space exhibits a distinct topology. Main parameters to tune:
- **min_dist**: Increasing the minimum distance between points caused the data points to spread further apart diluting finer topologies. 
- **n_neighbors**: A lower number of neighbours led to a more curled up structure while a higher number unfolded it. Increasing the parameter to more than 20 did not further change the embedding significantly.
- **n_components**: The number of dimensions of the embedding.

In [2]:
# Load data
df = pd.read_csv(f"{config.output_dir}wide_table_knn.csv")

# Scale data
minmax_scaler = MinMaxScaler().fit(df[config.parameters])
X_m = pd.DataFrame(minmax_scaler.transform(df[config.parameters]), columns=config.parameters)

# Robust scaler
robust_scaler = RobustScaler().fit(df[config.parameters])
X_r = pd.DataFrame(robust_scaler.transform(df[config.parameters]), columns=config.parameters)

X_dict = {"minmax": X_m, "robust": X_r}

In [ ]:
# # Iterate over combinations (and repeat combination until plot has correct rotation for final plot)
# embeddings = []
# for umap_kwargs in param_combos: 
#     print(f"min_dist =  {umap_kwargs["min_dist"]}\nn_components = {umap_kwargs["n_components"]}\nn_neighbors = {umap_kwargs["n_neighbors"]}\nmetric = {umap_kwargs["metric"]}")

#     # Compute and plot embedding
#     embedding = UMAP(**umap_kwargs).fit_transform(X)
#     umap_new = get_user_input(embedding)
#     # plot_embedding(umap_new, figsize=(4, 4), dpi=300, save_as=None)
    
#     # Append to results
#     embeddings.append({"embedding": [umap_new], 
#                        "min_dist": umap_kwargs["min_dist"], 
#                        "n_neighbors": umap_kwargs["n_neighbors"], 
#                        "metric": umap_kwargs["metric"], 
#                        "n_components": umap_kwargs["n_components"]})

In [ ]:
# # Save embeddings (for following plot)
# for info in embeddings:
#     e = info["embedding"]
#     n_neighbors = info["n_neighbors"]
#     min_dist = info["min_dist"]
#     metric = info["metric"]
#     n_components = info["n_components"]
#     np.save(f"{config.output_dir_clustering}embedding_n_neighbors{n_neighbors}_min_dist{min_dist}_metric{metric}_n_components{n_components}.npy", e)

# Compute metrics

In [3]:
def umap_grid_search(X_in, param_grid, subsample_size_metrics=2000, subsample_size_shepard=500, out_dir="umap_metrics_report", random_state=42, n_jobs=-1):
    """ Perform grid search over hyperparameters for UMAP, store results (diagrams and CSV) in the specified directory, and plot Shepard Diagram. """
    # Create output dir if it does not exist yet
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    # Prepare parameter combinations
    param_combinations = [dict(zip(param_grid.keys(), v)) for v in product(*param_grid.values())]

    # Subsample once for metrics
    X_metrics = resample(X_in, n_samples=subsample_size_metrics, random_state=random_state)

    def evaluate_umap(umap_hyps):  # n_neighbors, min_dist, metric):
        # Compute embedding
        umap_model = UMAP(**umap_hyps)
        embedding = umap_model.fit_transform(X_metrics)
        try:
            Xr = umap_model.inverse_transform(embedding)
        except:
            Xr = None  # Some UMAP configurations don't support inverse_transform

        # Compute metrics
        drm = DRMetrics(X_metrics, embedding, Xr)

        # Subsample for Shepard
        X_shepard = resample(X_in, n_samples=subsample_size_shepard, random_state=random_state)
        Z_shepard = umap_model.fit_transform(X_shepard)
        plot_name = f"{out_dir}/shepard_{'_'.join([f'{k}{v}' for k, v in umap_hyps.items()])}.png"
        plot_shepard(X_shepard, Z_shepard, save_as=plot_name)

        return {**umap_hyps, **{"Qlocal": drm.Qlocal, "Qglobal": drm.Qglobal, "AUC_T": drm.AUC_T, "AUC_C": drm.AUC_C}}

    # Parallel execution
    print(f"Running UMAP grid search on {len(param_combinations)} configurations...")
    results = Parallel(n_jobs=n_jobs)(
        delayed(evaluate_umap)(h) for h in tqdm(param_combinations, position=0, leave=True, desc="UMAP Evaluation")
    )

    # Save results
    df = pd.DataFrame(results)
    df.to_csv(os.path.join(out_dir, "umap_grid_metrics.csv"), index=False)

    print(f"Grid search complete. Results saved in '{out_dir}'\n")

    return df


In [4]:
def plot_shepard(X_sub, y_sub, save_as=None, dpi=300, figsize=(6, 6)):
    """ Plot Shepard diagram. """

    # Compute pairwise distances
    D_high = pairwise_distances(X_sub)
    D_low = pairwise_distances(y_sub)

    # Plot
    plt.figure(figsize=figsize)
    plt.scatter(D_high.flatten(), D_low.flatten(), alpha=0.1, s=1)
    plt.plot([D_high.min(), D_high.max()], [D_high.min(), D_high.max()], "r--")
    plt.xlabel("Original Distances")
    plt.ylabel("UMAP Distances")
    plt.title(f"Shepard Diagram")
    plt.tight_layout()
    if save_as:
        plt.savefig(save_as, dpi=dpi)
    plt.close()

    
def plot_qs(df_display, out_dir):
    """ Qlocal vs Qglobal plot. """
    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=df_display, x="Qglobal", y="Qlocal", hue="metric", style="min_dist", size="n_neighbors", palette="tab10")
    plt.xlabel("Qglobal")
    plt.ylabel("Qlocal")
    plt.title("UMAP Hyperparameter Grid: Qlocal vs Qglobal")
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(out_dir + "/q_scores.png")
    plt.show()


def plot_qs_interactive(df_display, out_dir):
    """ Interactive scatter Qlocal vs. Qglobal plot.  """
    fig = px.scatter(
        df_display,
        x="Qglobal",
        y="Qlocal",
        color="metric",
        size="n_neighbors",
        symbol="min_dist",
        title="Qlocal vs Qglobal by UMAP parameters",
        color_discrete_sequence=px.colors.qualitative.Set2,
    )
    
    # Update layout for axis labels and margins
    fig.update_layout(
        xaxis=dict(title="Qglobal"),
        yaxis=dict(title="Qlocal"),
        legend_title="Metric",
        margin=dict(l=40, r=40, t=40, b=40),
        width=1000, height=1000
    )
    
    fig.write_html(out_dir + "/q_scores.html")
    fig.show() 


In [ ]:
for exp_name, X in X_dict.items():
    print(exp_name)
    X = X.to_numpy()

    df_results = umap_grid_search(X,
        param_grid={
            "n_neighbors": [5, 10, 20, 30, 50, 100, 150],
            "min_dist":  [0.0, 0.001, 0.01, 0.05, 0.1, 0.3, 0.5, 0.8, 0.99],
            "metric": ["euclidean", "mahalanobis", "manhattan", "cosine", "chebyshev"],
            "n_components": [3]
        },
        subsample_size_metrics=2000,
        subsample_size_shepard=500,
        out_dir=f"{config.output_dir_umap}/{exp_name}",
        random_state=42,
        n_jobs=4  # Use -1 for all cores
    )
    

In [ ]:
# Plot Qlocal vs Qglobal 

# Iterate over scalers
for exp_name, X in X_dict.items():
    print(exp_name)
    df_results = pd.read_csv(f"{config.output_dir_umap}/{exp_name}/umap_grid_metrics.csv")

    # Plot Q
    plot_qs(df_results, f"{config.output_dir_umap}/{exp_name}/")
    plot_qs_interactive(df_results, f"{config.output_dir_umap}/{exp_name}/")


# Plotting with final hyperparameters

In [5]:
# Define final hyperparameter combinations
final_hyps = {"minmax": {"n_neighbors": 100, "min_dist": 0.5, "metric": "euclidean", "n_components": 3}, 
              "robust": {"n_neighbors": 20, "min_dist": 0.99, "metric": "euclidean", "n_components": 3}}

# Iterate over scalers
metrics = {}
for exp_name, hyps in final_hyps.items():
    print(exp_name)
    df_temp = df.copy()
    X_temp = X_dict[exp_name].to_numpy()

    # Compute embedding
    print("  Compute UMAP...")
    umap_model = UMAP(**hyps)
    embedding = umap_model.fit_transform(X_temp)

    # Inverse transform embedding
    print("  Compute inverse UMAP...")
    try:
        Xr = umap_model.inverse_transform(embedding)
    except:
        Xr = None  # Some UMAP configurations don't support inverse_transform
    
    # Compute metrics
    print("  Compute metrics...")
    drm = DRMetrics(X_temp, embedding, Xr)
    metrics[exp_name] = drm

    # Save embedding
    df_temp[["e0", "e1", "e2"]] = embedding
    df_temp.to_csv(f"{config.output_dir_umap}/{exp_name}/final_embeddings.csv")

    # Plot embedding
    print("  Plotting...")
    plot_embedding(embedding, save_as=f"{config.output_dir_umap}/{exp_name}/final_embedding.png", dpi=1000, figsize=(6, 6))
    plot_interactive_embedding(df_temp)

    # Plot Shepard Diagram
    plot_shepard(X_temp, embedding, save_as=f"{config.output_dir_umap}/{exp_name}/final_embedding_shepard.png", dpi=1000, figsize=(6, 6))

minmax
  Compute UMAP...
  Compute inverse UMAP...
  Compute metrics...


MemoryError: Unable to allocate 17.9 GiB for an array with shape (2403940900,) and data type float64